# Boost converter: verified state-space averaging

Ideal components, continuous-conduction mode, fixed switching frequency, and a resistive load are assumed.

In [1]:
from pathlib import Path
import sys
repository = Path.cwd()
if not (repository / 'python').exists():
    repository = repository.parent
sys.path.insert(0, str(repository / 'python'))
from elspice_mna.converters import analyze_converter, summary_text
analysis = analyze_converter('boost')
print(summary_text(analysis))


Boost converter
states: ['I(L1)', 'V(out)']
A = Matrix([[0, (D - 1)/L], [-(D - 1)/C, -1/(C*R)]])
B_g = Matrix([[1/L], [0]])
B_d = Matrix([[-V_g/(L*(D - 1))], [-V_g/(C*R*(D - 1)**2)]])
X_dc = Matrix([[V_g/(R*(D - 1)**2)], [-V_g/(D - 1)]])
G_vg(s) = -R*(D - 1)/(C*L*R*s**2 + D**2*R - 2*D*R + L*s + R)
G_vd(s) = -V_g*(-D**2*R + 2*D*R + L*s - R)/((D - 1)**2*(C*L*R*s**2 + D**2*R - 2*D*R + L*s + R))
verification: PASS


## State-space averaging result

The state order is $x=[i_L, v_o]^T$. The Rust parser and MNA builder produce
the two topology models below; SymPy then eliminates algebraic MNA variables.

$$A_{on}=\left[\begin{matrix}0 & 0\\0 & - \frac{1}{C R}\end{matrix}\right],\qquad
B_{g,on}=\left[\begin{matrix}\frac{1}{L}\\0\end{matrix}\right]$$

$$A_{off}=\left[\begin{matrix}0 & - \frac{1}{L}\\\frac{1}{C} & - \frac{1}{C R}\end{matrix}\right],\qquad
B_{g,off}=\left[\begin{matrix}\frac{1}{L}\\0\end{matrix}\right]$$

With $D'=1-D$:

$$\dot x=\left[\begin{matrix}0 & \frac{D - 1}{L}\\- \frac{D - 1}{C} & - \frac{1}{C R}\end{matrix}\right]x+\left[\begin{matrix}\frac{1}{L}\\0\end{matrix}\right]v_g$$

The DC operating point and duty perturbation vector are

$$X=\left[\begin{matrix}\frac{V_{g}}{R \left(D - 1\right)^{2}}\\- \frac{V_{g}}{D - 1}\end{matrix}\right],\qquad B_d=\left[\begin{matrix}- \frac{V_{g}}{L \left(D - 1\right)}\\- \frac{V_{g}}{C R \left(D - 1\right)^{2}}\end{matrix}\right].$$

Thus the input-to-output and control-to-output state-space models are

$$\dot{\hat x}=A\hat x+B_g\hat v_g,\quad
\hat v_o=[0\;1]\hat x,$$

$$\dot{\hat x}=A\hat x+B_d\hat d,\quad
\hat v_o=[0\;1]\hat x.$$

Their transfer functions are

$$G_{vg}(s)=- \frac{R \left(D - 1\right)}{C L R s^{2} + D^{2} R - 2 D R + L s + R},$$

$$G_{vd}(s)=- \frac{V_{g} \left(- D^{2} R + 2 D R + L s - R\right)}{\left(D - 1\right)^{2} \left(C L R s^{2} + D^{2} R - 2 D R + L s + R\right)}.$$


In [2]:
# Exact symbolic residual checks were executed by analyze_converter.
for check in analysis['verification']:
    print('PASS:', check)


PASS: Rust MNA phase matrices equal the hand-derived phase equations
PASS: Rust numeric Schur reduction equals the SymPy descriptor reduction
PASS: Averaged A, input vector, DC point, and duty vector equal hand derivation
PASS: Input-to-output and control-to-output transfer functions simplify exactly


## Independent verification

The output polarity is **positive**. These checks do not reuse the calculated
averaged matrices: textbook ON/OFF differential equations are entered
separately in `converters.py`, simplified, and compared element by element.

- Rust MNA phase matrices equal the hand-derived phase equations
- Rust numeric Schur reduction equals the SymPy descriptor reduction
- Averaged A, input vector, DC point, and duty vector equal hand derivation
- Input-to-output and control-to-output transfer functions simplify exactly

Every symbolic residual is exactly zero. Numeric phase matrices are also
evaluated independently by the Rust Schur-complement reducer and compared to
the SymPy result at $L=100\,\mu H$, $C=220\,\mu F$, and $R=12\,\Omega$.
